In [ ]:
# ---------------- Imports ----------------

import os
import json

import yaml
import numpy as np
import pandas as pd

from scipy.stats import spearmanr

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib as mpl

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.float_format", lambda x: f"{x:.3f}")

# Path to font
FONT_DIR = os.path.join("../../config", "fonts", "linux_libertine")
FONT_PATH = os.path.join(FONT_DIR, "LinLibertine_R.ttf")

# Register font
fm.fontManager.addfont(FONT_PATH)

# Get font name
libertine_font = fm.FontProperties(fname=FONT_PATH).get_name()

# Set globally
mpl.rcParams.update({
    "font.family": libertine_font,
    "pdf.fonttype": 42,
})



In [ ]:
# ---------------- Inputs ----------------

MODELS = {
    "L3":  "ibm-rank-30k/meta-llama/Llama-3.2-3B-Instruct/20260924t003019-100x100-k7-p0.5-s42-positiveonly-ociv-labelablation-trajectories.json",
    "L8":  "ibm-rank-30k/meta-llama/Llama-3.1-8B-Instruct/20260924t011840-100x100-k7-p0.5-s42-positiveonly-ociv-labelablation-trajectories.json",
    #"L70": "",
    "Q7":  "ibm-rank-30k/Qwen/Qwen2.5-7B-Instruct/20260924t025143-100x100-k7-p0.5-s42-positiveonly-ociv-labelablation-trajectories.json",
    "Q14": "ibm-rank-30k/Qwen/Qwen2.5-14B-Instruct/20260924t040340-100x100-k7-p0.5-s42-positiveonly-ociv-labelablation-trajectories.json",
    "Q72": "ibm-rank-30k/Qwen/Qwen2.5-72B-Instruct/20260924t215453-100x100-k7-p0.5-s42-positiveonly-ociv-labelablation-trajectories.json",
    "M8":  "ibm-rank-30k/mistralai/Ministral-3-8B-Instruct-2512/20260924t062830-100x100-k7-p0.5-s42-positiveonly-ociv-labelablation-trajectories.json",
    "M14": "ibm-rank-30k/mistralai/Ministral-3-14B-Instruct-2512/20260924t082847-100x100-k7-p0.5-s42-positiveonly-ociv-labelablation-trajectories.json",
    "M24": "ibm-rank-30k/mistralai/Mistral-Small-3.2-24B-Instruct-2506/20260924t103927-100x100-k7-p0.5-s42-positiveonly-ociv-labelablation-trajectories.json",
    "P4": "ibm-rank-30k/microsoft/Phi-4-mini-instruct/20260924t172718-100x100-k7-p0.5-s42-positiveonly-ociv-labelablation-trajectories.json",
    "G12": "ibm-rank-30k/google/gemma-4-12B-it/20260924t142014-100x100-k7-p0.5-s42-positiveonly-ociv-labelablation-trajectories.json",
}

In [ ]:
# ---------------- Config ----------------

with open("../../config/config.yaml", "r") as f:
    config = yaml.safe_load(f)

proj_store = config["paths"]["proj-store"]

input_folder = os.path.join(proj_store, "experiments", "ociv-trajectory")

OUTPUT_FOLDER = os.path.join(proj_store, "evaluation", "label-ablation")
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

OUTPUT_PATH = os.path.join(OUTPUT_FOLDER, "label-ablation-agreement")



In [ ]:
# ---------------- Utilities ----------------

def read_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def argmax_set(values):

    values = np.asarray(values, dtype=float)
    max_value = values.max()

    return set(np.flatnonzero(np.isclose(values, max_value)).tolist())


In [ ]:
# ---------------- Load matched scores ----------------

rows = []

for model_name, input_filename in MODELS.items():

    input_path = os.path.join(input_folder, input_filename)

    data = read_json(input_path)

    params = data.get("params", {})

    if not params.get("compute_label_ablation_scores", False):
        raise ValueError(
            f"{model_name} was not run with "
            f"--compute_label_ablation_scores."
        )

    for trajectory in data["trajectories"]:

        trajectory_id = trajectory["trajectory_id"]

        for step_record in trajectory["steps"]:

            step = step_record["step"]

            for candidate in step_record["candidates"]:

                if "positive_prob_with_history" not in candidate:
                    raise ValueError(
                        f"{model_name}, trajectory {trajectory_id}, "
                        f"step {step}: missing primary positive score."
                    )

                if "label_ablation" not in candidate:
                    raise ValueError(
                        f"{model_name}, trajectory {trajectory_id}, "
                        f"step {step}: missing label-ablation score."
                    )

                ablation = candidate["label_ablation"]

                if "positive_prob_with_history" not in ablation:
                    raise ValueError(
                        f"{model_name}, trajectory {trajectory_id}, "
                        f"step {step}: missing ablation positive score."
                    )

                rows.append({
                    "model": model_name,
                    "trajectory_id": trajectory_id,
                    "step": step,
                    "candidate_position": candidate["candidate_position"],
                    "argument_id": candidate["argument_id"],
                    "topic": candidate["topic"],
                    "selected_candidate": candidate["selected_candidate"],
                    "yes_score": float(
                        candidate["positive_prob_with_history"]
                    ),
                    "valuable_score": float(
                        ablation["positive_prob_with_history"]
                    ),
                })


scores_df = pd.DataFrame(rows)

print(
    f"{len(scores_df):,} matched candidate evaluations "
    f"across {scores_df['model'].nunique()} models."
)

display(scores_df.head())



In [ ]:
# ---------------- Candidate-set agreement ----------------

candidate_set_rows = []

group_columns = ["model", "trajectory_id", "step"]

for (
model_name,
trajectory_id,
step,
), group in scores_df.groupby(
    group_columns,
    sort=False,
):

    group = group.sort_values(
        "candidate_position"
    ).reset_index(drop=True)

    yes_scores = group["yes_score"].to_numpy()
    valuable_scores = group["valuable_score"].to_numpy()

    # ------------------------------------------------------------
    # Rank agreement within this exact candidate set
    # ------------------------------------------------------------

    rho = spearmanr(
        yes_scores,
        valuable_scores,
    ).statistic

    # ------------------------------------------------------------
    # Counterfactual local argmax agreement
    # ------------------------------------------------------------

    yes_argmax = argmax_set(yes_scores)
    valuable_argmax = argmax_set(valuable_scores)

    exact_argmax_agreement = (yes_argmax == valuable_argmax)

    argmax_overlap = bool(yes_argmax & valuable_argmax)

    # The actual selected candidate followed the Yes policy.
    selected_positions = set(group.loc[group["selected_candidate"], "candidate_position"].astype(int))

    if len(selected_positions) != 1:
        raise ValueError(
            f"{model_name}, trajectory {trajectory_id}, "
            f"step {step}: expected exactly one selected candidate, "
            f"found {len(selected_positions)}."
        )

    actual_selected_position = next(iter(selected_positions))

    valuable_would_allow_actual_selection = (actual_selected_position in valuable_argmax)

    candidate_set_rows.append({
        "model": model_name,
        "trajectory_id": trajectory_id,
        "step": step,
        "candidate_k": len(group),
        "spearman_rho": rho,
        "yes_n_argmax": len(yes_argmax),
        "valuable_n_argmax": len(valuable_argmax),
        "exact_argmax_agreement": exact_argmax_agreement,
        "argmax_overlap": argmax_overlap,
        "valuable_would_allow_actual_selection": (
            valuable_would_allow_actual_selection
        ),
    })


candidate_set_df = pd.DataFrame(candidate_set_rows)

print()
print("Candidate-set-level agreement")
display(candidate_set_df.head())



In [ ]:
# ---------------- Summarize by model ----------------

summary_rows = []

for model_name, group in candidate_set_df.groupby("model", sort=False):

    valid_rhos = group["spearman_rho"].dropna()

    summary_rows.append({
        "model": model_name,

        "n_candidate_sets": len(group),
        "n_spearman_defined": len(valid_rhos),
        "fraction_spearman_defined": (
            len(valid_rhos) / len(group)
        ),

        "mean_spearman_rho": (
            valid_rhos.mean()
            if len(valid_rhos)
            else np.nan
        ),

        "median_spearman_rho": (
            valid_rhos.median()
            if len(valid_rhos)
            else np.nan
        ),

        "std_spearman_rho": (
            valid_rhos.std(ddof=1)
            if len(valid_rhos) > 1
            else np.nan
        ),

        "exact_argmax_agreement": (group["exact_argmax_agreement"].mean()),

        "argmax_overlap": (group["argmax_overlap"].mean()),

        "valuable_would_allow_actual_selection": (group["valuable_would_allow_actual_selection"].mean()),

        "yes_tie_rate": (group["yes_n_argmax"].gt(1).mean()),

        "valuable_tie_rate": (group["valuable_n_argmax"].gt(1).mean()),
    })


summary_df = pd.DataFrame(summary_rows)




In [ ]:
# ---------------- raw-score correlation ----------------

raw_score_rows = []

for model_name, group in scores_df.groupby("model", sort=False):

    rho = spearmanr(group["yes_score"], group["valuable_score"]).statistic

    raw_score_rows.append({"model": model_name, "n_candidates": len(group), "global_spearman_rho": rho})


raw_score_df = pd.DataFrame(raw_score_rows)



In [ ]:
# ---------------- Display ----------------

print()
print("Matched Yes vs Valuable agreement")
print()

display(summary_df.sort_values("mean_spearman_rho", ascending=False).reset_index(drop=True))

print()
print("Global score correlation (secondary)")
print()

display(raw_score_df.sort_values("global_spearman_rho", ascending=False).reset_index(drop=True))



In [ ]:
# ---------------- Plot mean within-set Spearman correlation ----------------
plt.rcParams.update({"font.size": 22})


plot_df = summary_df.sort_values("mean_spearman_rho", ascending=False).reset_index(drop=True)


fig, ax = plt.subplots(figsize=(8, 4))


ax.bar(
    plot_df["model"],
    plot_df["mean_spearman_rho"],
    color="#004C80",
)

ax.set_xlabel("Model")
ax.set_ylabel("Mean within-set $\\rho$")
ax.set_ylim(0, 1)

fig.tight_layout()

fig.savefig(f"{OUTPUT_PATH}-mean-spearman-rho.pdf", bbox_inches="tight")

plt.show()

In [ ]:
# ---------------- Save ----------------

scores_df.to_csv(f"{OUTPUT_PATH}-candidate-scores.csv", index=False)

candidate_set_df.to_csv(f"{OUTPUT_PATH}-candidate-sets.csv", index=False)

summary_df.to_csv(f"{OUTPUT_PATH}-summary.csv", index=False)

raw_score_df.to_csv(f"{OUTPUT_PATH}-global-score-correlation.csv", index=False)

print()
print(f"Saved results to {OUTPUT_FOLDER}")

